# Conduct permutation test

**Notebook contents:**
* Obtains permutation test data
* Runs the permutation test and extracts corresponding p-values
* Graphs the p-values

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import utils
from utils import set_plot_style
from main import DorsognaTE

# Preliminaries

In [ ]:
# list of models

phenotype_dict = {
    'singlemill':[0.5,0.1],
    'doublemill':[0.9,0.5],
    'doublering':[0.5,0.5],
    'collswarm':[0.5,0.9],
    'escapesymm':[2,0.9],
    'escapeunsymm':[2,2],
    'escapecoll':[2,3]
}

legend_labels = utils.proper_phenotype_names

versions = ['linvel', 'angvel']

# Run permutation test

In [ ]:
output_dir = 'csvs_consolidated_permute_data'

for phenotype_name, values in phenotype_dict.items():
    C, l = values
    os.makedirs(output_dir, exist_ok=True)
    insights_path = os.path.join(output_dir, phenotype_name)
    model = DorsognaTE(outdir=insights_path)
    model.develop_model(
            C=C,
            l=l,
            phenotype_name=phenotype_name,
            particle_count=100,
            t_max=40,
            vel_scale=0.1,
            fps=20,
            dt=0.1,
            animate=False
        )
    for permute_seed in range(1,101):
        for TE_ver in ['linvel','angvel']:
            model.compute_modelTE(TE_ver = TE_ver,
                        TE_embedding = 15,
                        permute_seed = permute_seed)

# Permutation test result exports

## Data exports
The test uses 100 permuted time series setups which was not included in the GitHub repository due to file size constraints.

In [ ]:
t = np.linspace(0,40,400)

folder_name = "csvs_consolidated_permute_data"

if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder {folder_name} created")

for ver in versions:
    for phenotype in phenotype_dict.keys():
        phenotype_name = phenotype
        coeff = phenotype_dict[phenotype]

        # actual TE values
        filename = f"csvs_actual_te_values/{phenotype_name}/TElog_{phenotype_name}_{coeff[0]}_{coeff[1]}_{ver}_k15.csv"
        data_df = pd.read_csv(filename, index_col = [0])
        permute_data_df = pd.DataFrame()

        for i in range(1,101):
            df_perm = pd.read_csv(f'csvs_permutation_tests/{phenotype_name}_permute/{phenotype_name}_permute_{ver}/TElog_{phenotype_name}_{coeff[0]}_{coeff[1]}_{ver}_k15_permuted{i}.csv', index_col = [0])
            ts_perm = df_perm.mean(axis=1)
            ts_perm_smooth = ts_perm.rolling(window=10, center= True).mean()
            permute_data_df[i] = ts_perm_smooth
        
        # save
        file_name = f"{phenotype_name}_{ver}_consolidated_permute_data.csv"
       
        save_path = os.path.join(folder_name, file_name)
        permute_data_df.to_csv(save_path)
        print(f"Saved CSV: {file_name}")


In [ ]:
permute_dict = dict()

for ver in versions:
    for phenotype in phenotype_dict.keys():
        phenotype_name = phenotype
        coeff = phenotype_dict[phenotype]

        folder_name = "csvs_consolidated_permute_data"
        file_name = f"{phenotype_name}_{ver}_consolidated_permute_data.csv"
        save_path = os.path.join(folder_name, file_name)
        
        permute_data_df = pd.read_csv(save_path, index_col=0)
        permute_dict[phenotype, ver] = permute_data_df

permute_dict.keys()


## Plotting

In [ ]:
for ver in versions:
    for phenotype in phenotype_dict.keys():
        phenotype_name = phenotype
        coeff = phenotype_dict[phenotype]
        permute_data_df = permute_dict[phenotype, ver]

        df_true = pd.read_csv(f'csvs_actual_te_values\{phenotype_name}\TElog_{phenotype_name}_{coeff[0]}_{coeff[1]}_{ver}_k15.csv', index_col = 0)
        ts_true = df_true.mean(axis=1)
        
        # pvalues
        df_pvals = permute_data_df.ge(ts_true.values, axis =0).astype(int).mean(axis=1)

        # save
        folder_name = "csvs_pvalues"
        file_name = f"{phenotype_name}_{ver}_pvals.csv"

        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
            print(f"folder {folder_name} created")

        save_path = os.path.join(folder_name, file_name)
        df_pvals.to_csv(save_path)
        print(f"Saved CSV: {file_name}")

        df_pvals = pd.read_csv(f"csvs_pvalues\{phenotype_name}_{ver}_pvals.csv", index_col=0)

        # PLOT 1: P VALUES 
        t = np.linspace(0,40,400)

        set_plot_style()

        plt.figure(figsize=(8, 4)) 
        plt.plot(t, df_pvals, color="black", linewidth=0.8)

        # zero line
        plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")
        plt.axhline(y=0.10, color = "red", linewidth = 0.6, linestyle='--', label = 'p-value = 0.1')

        # axis labels
        plt.xlabel("Time Step ($t$)")
        plt.ylabel("p-value")

        # bounds
        plt.xlim(0, 40)
        plt.ylim(-0.05, 1.15)
        plt.grid()
        
        # legend
        plt.legend(bbox_to_anchor=(1.01, -0.17), loc='lower right', fontsize=9)
        plt.tight_layout()

        # save
        folder_name = "graphs/pvalues"
        file_name = f"{phenotype_name}_{ver}_pvalues.png"

        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
            print(f"folder {folder_name} created")

        save_path = os.path.join(folder_name, file_name)

        plt.savefig(save_path, dpi=300, bbox_inches="tight")

        plt.show()

        # PLOT 2: OVERALL PERMUTE VS ACTUAL ALL
        overall_permute_ts = permute_data_df.mean(axis=1)
        set_plot_style()

        plt.figure(figsize=(8, 4)) 
        for i in range(1, 101):
            plt.plot(t, permute_data_df[f"{i}"], color="grey", alpha=0.1, linewidth=0.4)

        plt.plot(t, ts_true, color="blue", linewidth=0.8, label="Observed $TE$")
        plt.plot(t, overall_permute_ts, color="red", linewidth=1, label="Average $TE$ of Permutations")

        # show significant times
        significant = df_pvals.values.flatten() < 0.10

        plt.fill_between(t, 0, 1, where=significant, alpha=0.30, color='orange', transform=plt.gca().get_xaxis_transform(), label='p < 0.10')

        # zero line
        plt.axhline(0, color="gray", linewidth=0.6, linestyle=":")

        # axis labels
        plt.xlabel("Time Step ($t$)")
        plt.ylabel("Local Transfer Entropy ($TE$)")

        # bounds
        plt.xlim(0, 40)
        plt.grid()
        
        # legend
        plt.legend(bbox_to_anchor=(1.02, -0.25), loc='lower right', ncol=2, fontsize=9)
        plt.tight_layout()


        # save
        folder_name = "graphs/permute_vs_trueactual_pvalue"
        file_name = f"{phenotype_name}_{ver}_permute_vs_trueactual_pvalue.png"

        if not os.path.exists(folder_name):
            os.makedirs(folder_name)
            print(f"folder {folder_name} created")

        save_path = os.path.join(folder_name, file_name)

        plt.savefig(save_path, dpi=300, bbox_inches="tight")

        plt.show()


# Subplot view

In [ ]:
from matplotlib.gridspec import GridSpec

t = np.linspace(0,40,400)

set_plot_style()


for ver in versions:
    fig = plt.figure(figsize=(18, 8))
    gs = GridSpec(2, 8, figure=fig)  

    ax0 = fig.add_subplot(gs[0, 0:2])
    ax1 = fig.add_subplot(gs[0, 2:4])
    ax2 = fig.add_subplot(gs[0, 4:6])
    ax3 = fig.add_subplot(gs[0, 6:8])
    ax4 = fig.add_subplot(gs[1, 1:3])
    ax5 = fig.add_subplot(gs[1, 3:5])
    ax6 = fig.add_subplot(gs[1, 5:7])

    axes_flat = [ax0, ax1, ax2, ax3, ax4, ax5, ax6]

    for plot_idx, phenotype in enumerate(phenotype_dict.keys()):
        phenotype_name = phenotype
        coeff = phenotype_dict[phenotype]
        permute_data_df = permute_dict[phenotype, ver]

        df_true = pd.read_csv(f'csvs_actual_te_values\{phenotype_name}\TElog_{phenotype_name}_{coeff[0]}_{coeff[1]}_{ver}_k15.csv', index_col=0)
        ts_true = df_true.mean(axis=1)

        df_pvals = permute_data_df.ge(ts_true.values, axis=0).astype(int).mean(axis=1)

        overall_permute_ts = permute_data_df.mean(axis=1)
        significant = df_pvals.values.flatten() < 0.10

        ax = axes_flat[plot_idx]

        # permutation lines
        for i in range(1, 101):
            ax.plot(t, permute_data_df[f"{i}"], color="grey", alpha=0.1, linewidth=0.4)

        # main lines
        ax.plot(t, ts_true, color="blue", linewidth=0.8, label="Observed $TE$")
        ax.plot(t, overall_permute_ts, color="red", linewidth=1, label="Average $TE$ of Permutations")

        # shading
        ax.fill_between(t, 0, 1, where=significant, alpha=0.30, color='orange', transform=ax.get_xaxis_transform(), label='p < 0.10')

        # zero line
        ax.axhline(0, color="gray", linewidth=0.6, linestyle=":")
        ax.set_title(legend_labels[phenotype], fontsize = 15)
        ax.set_xlim(0, 40)
        ax.grid()

        if plot_idx == len(axes_flat) - 1:
            handles, labels = ax.get_legend_handles_labels()

    for idx in range(len(phenotype_dict), len(axes_flat)):
        axes_flat[idx].set_visible(False)

    fig.supylabel("Local Transfer Entropy ($TE$)", fontsize=20, y=0.50)
    fig.supxlabel("Time Step ($t$)", fontsize=20)
    fig.legend(handles, labels, bbox_to_anchor=(0.99, 0), loc="lower right", ncol=3, fontsize=12)

    plt.tight_layout(rect=[0.01, 0, 1, 0.99])

    folder_name = "graphs/permute_vs_trueactual_pvalue"
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    save_path = os.path.join(folder_name, f"all_subplots_{ver}_permute_vs_trueactual_pvalue.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()